In [ ]:
import tomllib
import pandas as pd
import hanzidentifier
from opencc_purepy import OpenCC
import jieba


cc_t2tw = OpenCC('t2tw')
cc_s2tw = OpenCC('s2tw')

def load_data(config: dict) -> pd.DataFrame:
    df = pd.read_csv(
        f'data/{config['info']['name']}.csv',
        sep=config['sep'][0],
        names=['char', *config['columns']]
    )

    def parse_parentheses(text):
        if config['com'][0] in text and config['com'][1] in text:
            return text[:-1].split(config['com'][0])
        return text, ''

    for column in config['columns']:
        df[column] = df[column].str.strip()
        if df[column].str.contains(config['sep'][1]).any():
            df[column] = df[column].str.split(config['sep'][1])
        if df.loc[df[column].notna(), column].map(
            lambda x: any(
                config['com'][0] in x
                for x in x
            )
        ).any():
            df.loc[
                df[column].notna(), column
            ] = df.loc[
                df[column].notna(), column
            ].map(lambda x: x and {
                key: value
                for key, value in zip(
                    ['note', 'comment'],
                    zip(*map(parse_parentheses, x))
                )
            } or x)

    df = df[df.char.map(
        lambda x: hanzidentifier.identify(x) != 2
    )].reset_index(drop=True)

    df['char'] = df.char.map(lambda x:
        (x_tw := cc_t2tw.convert(x)) != x and
        x_tw not in df.char.values and
        x_tw or x
    )
    return df

def get_value(df: pd.DataFrame, char: str, column: str) -> str | None:
    def get_value_(df: pd.DataFrame, char: str, column: str) -> str | None:
        row = df[df.char == char]
        if not row.empty:
            return row.iloc[0][column] or None
        return None
    value: str | None = get_value_(df, char, column)
    if value is None:
        char = cc_s2tw.convert(char)
        value: str | None = get_value_(df, char, column)
    return value

def get_note(df: pd.DataFrame, char: str, column: str) -> dict[str, list[str] | None]:
    value: str | None = get_value(df, char, column)
    if value is None:
        if len(char) > 1:
            return {char: get_value(df, char, column) for char in char}
        char = cc_s2tw.convert(char)
        value: str | None = get_value(df, char, column)
    if len(char) > 1:
        return {char: [value] for char, value in zip(char, value)}
    return {char: value}
    

def get_note_with_comment(df: pd.DataFrame, char: str, column: str) -> dict[str, list[str, str] | None] | None:
    value: str | None = get_value(df, char, column)
    if len(char) > 1:
        return {char: get_value(df, char, column) for char in char}
    return {char: value}

def get_function(df: pd.DataFrame, column: str) -> callable[
    [pd.DataFrame, str, str],
    dict[
        str,
        list[str] | dict[
            str,
            list[str] | None
        ] | None
    ]
]:
    if df[column].map(lambda x: isinstance(x, dict)).any():
        return get_note_with_comment
    return get_note

def get_result(lang_name: str, scheme_name: str, text: str) -> dict[
    str,
    list[str] | dict[
        str,
        list[str] | None
    ] | None
]:
    config = tomllib.load(open(f'data/{lang_name}.config.toml', 'rb'))
    df = load_data(config)
    function = get_function(df, scheme_name)
    return {char: value for char in jieba.cut(text, cut_all=False) for char, value in function(df, char, scheme_name).items()}

# 用例

# 將「設置方案」中的方案列表顯示為「音系名・方案名」，例如「粵語・粵拼」「通字・通字羅馬字」
# 勾選的方案對應一個元組：(音系名 lang_name（如國語、粵語、通字等），方案名 scheme_name（如粵拼、通字羅馬字、通字漢字等）)
lang_name = '通字'  # lang_name 是勾選項目的音系名，亦是文件名
scheme_name = '通字羅馬字' # scheme_name 是勾選項目的方案名，即表的字段名
text = '作爲見着你，' # 輸入框文本
get_result(lang_name, scheme_name, text) # 將 get_result 函數綁定到某一路由

scheme_name = '通字漢字'
get_result(lang_name, scheme_name, text) # get_result 函數有兩種返回值，若值為字典，則代表有註解(comment)，若值為列表，則代表無註解，有註解需要在輸入框上方顯示勾選「註解」，註解標在漢字正下方。comment 與 note 一一對應

{'作': {'note': ('zoc',), 'comment': ('',)},
 '爲': {'note': ('wui', 'wuy'), 'comment': ('', '爲去')},
 '見': {'note': ('cienn',), 'comment': ('',)},
 '着': {'note': ('dyuh',), 'comment': ('',)},
 '你': {'note': ('nii',), 'comment': ('',)},
 '，': None}